# Prepare SQuAD_tiny Dataset for Assignment 2

This code prepare SQuAD_tiny from the SQuAD dataset. 

# 0. Import libraries

In [2]:
#!pip install datasets

In [4]:
#!pip install transformers rouge-score

In [6]:
#!pip install sentencepiece

In [8]:
#pip install 'accelerate>=0.26.0'

In [9]:
import os
import torch
import shutil
import gc
import numpy as np
from datasets import load_dataset
from transformers import T5Tokenizer, T5ForConditionalGeneration, Trainer, TrainingArguments, DataCollatorForSeq2Seq, EarlyStoppingCallback
from rouge_score import rouge_scorer
from tqdm import tqdm
from transformers import logging as transformers_logging
from itertools import batched
import pandas as pd
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

In [10]:
# Set seed for reproducibility
torch.manual_seed(42)


# 1. Load and preprocess SQuAD dataset

In [11]:
# 1. Load and preprocess SQuAD dataset
dataset = load_dataset("squad")

README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

plain_text/validation-00000-of-00001.par(…):   0%|          | 0.00/1.82M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

In [12]:
# Take subsets to avoid overload
train_dataset = dataset["train"].select(range(10000,11000))
val_dataset = dataset["validation"].select(range(3000,3100))
test_dataset = dataset["validation"].select(range(3100, 3200))  # No official SQuAD test set

In [13]:
print("Size of training set:", len(train_dataset))
print("Size of validation set:", len(val_dataset))
print("Size of testing set:", len(test_dataset))

Size of training set: 1000
Size of validation set: 100
Size of testing set: 100


In [14]:
MODEL_NAME = "t5-small"
#MODEL_NAME = "t5-base"
MAX_INPUT_LENGTH = 512
MAX_OUTPUT_LENGTH = 128
# Load tokenizer and model
tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME)
model = T5ForConditionalGeneration.from_pretrained(MODEL_NAME)

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [15]:
def encode_question_and_context(question, context):
    return f"question: {question}  context: {context}"

# Obtains the context, question and answer from a given sample.
def extract_sample_parts(sample):
    context = sample["context"]
    question = sample["question"]
    answer = sample["answers"]['text'][0]
    question_with_context = encode_question_and_context(question, context)
    return (question_with_context, question, answer)

# Encodes the sample, returning token IDs.
def preprocess(sample):
    # Extract data from sample.
    question_with_context, question, answer = extract_sample_parts(sample)

    # Using truncation causes the tokenizer to emit a warning for every sample.
    # This will generate a significant amount of messages, and likely crash
    # your browser tab. We temporarily disable log messages to work around this.
    # See https://github.com/huggingface/transformers/issues/14285
    old_level = transformers_logging.get_verbosity()
    transformers_logging.set_verbosity_error()
    
    # Generate tokens for the input.
    # We include both the context and the question (first two parameters).
    input_tokens = tokenizer(question_with_context, question, padding="max_length",
                             truncation=True, max_length=MAX_INPUT_LENGTH)

    # Generate tokens for the expected answer. There is no need to include the 
    output_tokens = tokenizer(answer, padding="max_length", truncation=True,
                              max_length=MAX_OUTPUT_LENGTH)

    # Restore old logging level, see above.
    transformers_logging.set_verbosity(old_level)

    # The output of the tokenizer is a map containing {input_ids, attention_mask}.
    # For trianing, we need to add the labels (answer/output tokens) to the map.
    input_tokens["labels"] = np.array(output_tokens["input_ids"])

    return input_tokens

In [16]:
# Preprocess the datasets
training_set_enc = train_dataset.map(preprocess, batched=False)
validation_set_enc = val_dataset.map(preprocess, batched=False)
testing_set_enc = test_dataset.map(preprocess, batched=False)

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Save the datasets to use them in task5.ipynb

2. Fine-Tuning the T5 model

In [17]:
# Use PyTorch tensors to improve the performance of the training process
cols = ["input_ids", "attention_mask", "labels"]
training_set_enc.set_format(type="torch", columns=cols)
validation_set_enc.set_format(type="torch", columns=cols)
testing_set_enc.set_format(type="torch", columns=cols)

In [18]:
# to use less memory
model.config.use_cache = False

To prevent overfitting we used Early Stopping

In [19]:
# Avoid conflicts with an already existing results directory
if os.path.exists("./results"):
    shutil.rmtree("./results")
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

model.config.use_cache = False

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    pad_to_multiple_of=8
)

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=30,                  
    
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    fp16=torch.cuda.is_available(),
    gradient_checkpointing=True,
    eval_accumulation_steps=8,
    dataloader_num_workers=0,
    
    eval_strategy="epoch",                
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    learning_rate=1e-4,
    weight_decay=0.01,
    save_total_limit=2,
    logging_dir="./logs",
    logging_steps=10,
    overwrite_output_dir=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=training_set_enc,
    eval_dataset=validation_set_enc,
    tokenizer=tokenizer,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)]
)
trainer.train(resume_from_checkpoint=False)

/tmp/ipykernel_1358/1832176544.py:42: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
/opt/conda/lib/python3.12/site-packages/transformers/data/data_collator.py:741: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:254.)
  batch["labels"] = torch.tensor(batch["labels"], dtype=torch.int64)


Epoch,Training Loss,Validation Loss
1,0.118200,0.071445
2,0.024000,0.021100
3,0.019900,0.020924
4,0.016600,0.020308
5,0.012800,0.021668
6,0.012000,0.021705
7,0.015700,0.023966
8,0.011600,0.022760
9,0.010700,0.024127


There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


TrainOutput(global_step=1125, training_loss=0.20698019218444824, metrics={'train_runtime': 710.5461, 'train_samples_per_second': 42.221, 'train_steps_per_second': 5.278, 'total_flos': 1218076213248000.0, 'train_loss': 0.20698019218444824, 'epoch': 9.0})

Evaluation on the test set

In [20]:
def compute_average_score(scores, metric, key):
     total = 0
     for i in range(len(scores)):
         total += getattr(scores[i][metric], key)
     return total / len(scores)

def compute_rouge(predictions, references):
    metrics = ["rouge1", "rouge2", "rougeL"]
    
    scorer = rouge_scorer.RougeScorer(metrics, use_stemmer=True)
    
    scores = []
    for prediction, reference in zip(predictions, references):
        scores.append(scorer.score(prediction, reference))

    results = {}
    for metric in metrics:
        for k in ["precision", "recall", "fmeasure"]:
            results[f"{metric}_{k}"] = compute_average_score(
                scores, metric, k)
    return results

In [21]:
model.eval()
answers_ctx, refs_ctx = [], []

for ex in test_dataset: 
    q, c = ex["question"], ex["context"]
    ref = ex["answers"]["text"][0] if ex["answers"]["text"] else ""
    inp = encode_question_and_context(q, c)

    toks = tokenizer(inp, return_tensors="pt", truncation=True, max_length=512)
    if torch.cuda.is_available():
        toks = {k: v.to("cuda") for k, v in toks.items()}

    with torch.no_grad():
        out = model.generate(**toks, max_new_tokens=64, num_beams=4, early_stopping=True)

    pred = tokenizer.decode(out[0], skip_special_tokens=True).strip()
    answers_ctx.append(pred.replace("\n", " "))
    refs_ctx.append(ref.replace("\n", " "))

In [22]:
rouge_ctx = compute_rouge(answers_ctx, refs_ctx)
print("ROUGE (with context):", rouge_ctx)

ROUGE (with context): {'rouge1_precision': 0.7147838827838826, 'rouge1_recall': 0.7508333333333335, 'rouge1_fmeasure': 0.7124629912571088, 'rouge2_precision': 0.47408333333333325, 'rouge2_recall': 0.517, 'rouge2_fmeasure': 0.47967460317460314, 'rougeL_precision': 0.7147838827838826, 'rougeL_recall': 0.7508333333333335, 'rougeL_fmeasure': 0.7124629912571088}


Generative Analysis on Test Samples

In [23]:
def generate_response(tokenizer, model, question):
    tokenized = tokenizer(question, return_tensors="pt", padding=True, truncation=True, max_length=MAX_OUTPUT_LENGTH).to(model.device)
    with torch.no_grad():
        outputs = model.generate(**tokenized)
    outputs = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    return outputs

def generate_answers(tokenizer, model, dataset, use_context=True, limit=None):
    if limit is not None:
        dataset = dataset.select(range(limit))
    questions = []
    inputs = []
    references = []
    for sample in dataset:
        question_with_context, question, answer = extract_sample_parts(sample)
        if use_context:
            inputs.append(question_with_context)
        else:
            inputs.append(question)
        questions.append(question)
        references.append(answer)
    outputs = []
    for samples in batched(inputs, 128):
        responses = generate_response(tokenizer, model, list(samples))
        outputs.extend(responses)
    assert (len(outputs) == len(references))
    return outputs, references, questions
     

In [24]:
# Generate answers with context
answers_ctx, refs_ctx, questions_ctx = generate_answers(
    tokenizer, model, test_dataset, use_context=True, limit=5
)
# Generate answers without context
answers_noctx, refs_noctx, questions_noctx = generate_answers(
    tokenizer, model, test_dataset, use_context=False, limit=5
)

In [25]:
for i, (q, gold, pred_ctx, pred_q) in enumerate(zip(questions_ctx, refs_ctx, answers_ctx, answers_noctx)):
    print(f"=== Test sample {i} ===")
    print("Question:", q)
    print("Gold answer:", gold)
    print("Pred (with context):", pred_ctx)
    print("Pred (question only):", pred_q)
    print()

=== Test sample 0 ===
Question: What country initially received the largest number of Huguenot refugees?
Gold answer: the Dutch Republic
Pred (with context): Dutch Republic
Pred (question only): 

=== Test sample 1 ===
Question: How many refugees emigrated to the Dutch Republic?
Gold answer: an estimated total of 75,000 to 100,000 people
Pred (with context): 2 million
Pred (question only): Wie viele Flüchtlinge emigrierten in die Niederlande?

=== Test sample 2 ===
Question: What was the population of the Dutch Republic before this emigration?
Gold answer: ca. 2 million
Pred (with context): ca. 2 million
Pred (question only): 

=== Test sample 3 ===
Question: What two areas in the Republic were first to grant rights to the Huguenots?
Gold answer: Amsterdam and the area of West Frisia
Pred (with context): the village of Fraissinet-de-Lozère
Pred (question only): were the first to grant rights to the Huguenots?

=== Test sample 4 ===
Question: What declaration predicated the emigration o

Comparison with a pretrained model

In [29]:
# a. Load Pre-trained model
# PRETRAINED_ID = "mrm8488/t5-base-finetuned-squadv2" (checkpoints of this model weren't loaded correctly)
PRETRAINED_ID = "MaRiOrOsSi/t5-base-finetuned-question-answering"
pre_tok = AutoTokenizer.from_pretrained(PRETRAINED_ID)
pre_model = AutoModelForSeq2SeqLM.from_pretrained(PRETRAINED_ID)
device = "cuda" if torch.cuda.is_available() else "cpu"
pre_model.to(device)
# pre_model.eval()

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/892M [00:00<?, ?B/s]

T5ForConditionalGeneration(
  (shared): Embedding(32128, 768)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 768)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=768, out_features=768, bias=False)
              (k): Linear(in_features=768, out_features=768, bias=False)
              (v): Linear(in_features=768, out_features=768, bias=False)
              (o): Linear(in_features=768, out_features=768, bias=False)
              (relative_attention_bias): Embedding(32, 12)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=768, out_features=3072, bias=False)
              (wo): Linear(in_features=3072, out_features=768, bias=False)
              (dropout): Dro

model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

In [30]:
# b. Replicate the evaluation steps from Tasks 3 and 4
pre_answers_ctx, pre_refs_ctx = [], []

for ex in test_dataset: 
    q, c = ex["question"], ex["context"]
    ref = ex["answers"]["text"][0] if ex["answers"]["text"] else ""
    inp = encode_question_and_context(q, c)

    toks = pre_tok(inp, return_tensors="pt", truncation=True, max_length=512)
    toks = {k: v.to(device) for k, v in toks.items()}

    with torch.no_grad():
        out = pre_model.generate(**toks, max_new_tokens=64, num_beams=4, early_stopping=True)

    pred = pre_tok.decode(out[0], skip_special_tokens=True).strip()
    pre_answers_ctx.append(pred.replace("\n", " "))
    pre_refs_ctx.append(ref.replace("\n", " "))

In [31]:
pre_rouge_ctx = compute_rouge(pre_answers_ctx, pre_refs_ctx)
print("ROUGE on pretrained Model (test, with context):", pre_rouge_ctx)

ROUGE on pretrained Model (test, with context): {'rouge1_precision': 0.7764624542124543, 'rouge1_recall': 0.8439880952380951, 'rouge1_fmeasure': 0.7840716783216783, 'rouge2_precision': 0.4585555555555555, 'rouge2_recall': 0.5106666666666667, 'rouge2_fmeasure': 0.46921300921300924, 'rougeL_precision': 0.7750338827838829, 'rougeL_recall': 0.8406547619047618, 'rougeL_fmeasure': 0.7820716783216785}


In [32]:
# c. Compare its performance against your fine-tuned SQuAD_tiny model in a table 

tiny = rouge_ctx        
pre  = pre_rouge_ctx    

cols = [
    "rouge1_precision","rouge1_recall","rouge1_fmeasure",
    "rouge2_precision","rouge2_recall","rouge2_fmeasure",
    "rougeL_precision","rougeL_recall","rougeL_fmeasure"
]

df = pd.DataFrame.from_dict(
    {
        "SQuAD_tiny": {k: tiny.get(k, float("nan")) for k in cols},
        "Pretrained": {k: pre.get(k, float("nan")) for k in cols},
    },
    orient="index"
)[cols]

df = df.rename(columns={
    "rouge1_precision":"R1 Prec", "rouge1_recall":"R1 Rec", "rouge1_fmeasure":"R1 F1",
    "rouge2_precision":"R2 Prec", "rouge2_recall":"R2 Rec", "rouge2_fmeasure":"R2 F1",
    "rougeL_precision":"RL Prec", "rougeL_recall":"RL Rec", "rougeL_fmeasure":"RL F1",
})

df.round(3)


,R1 Prec,R1 Rec,R1 F1,R2 Prec,R2 Rec,R2 F1,RL Prec,RL Rec,RL F1
SQuAD_tiny,0.715,0.751,0.712,0.474,0.517,0.480,0.715,0.751,0.712
Pretrained,0.776,0.844,0.784,0.459,0.511,0.469,0.775,0.841,0.782


In [ ]:
pre_answers_ctx 

In [30]:
pre_refs_ctx

['the Dutch Republic',
 'an estimated total of 75,000 to 100,000 people',
 'ca. 2 million',
 'Amsterdam and the area of West Frisia',
 'the revocation of the Edict of Nantes',
 'Tours',
 'Huguon',
 'the ghost of le roi Huguet',
 'prétendus réformés',
 'night',
 'Canterbury',
 'The Weavers',
 'economic separation',
 'Kent, particularly Sandwich, Faversham and Maidstone',
 'a restaurant',
 'Cork City',
 'Dublin, Cork, Youghal and Waterford',
 'Dublin',
 'a High Sheriff and one of the founders of the Bank of Ireland',
 '1696',
 'brain drain',
 'New France',
 'non-Catholics',
 "Seven Years' War",
 '1759-60',
 'Henry of Navarre',
 '1598',
 'granted the Protestants equality with Catholics',
 'the founding of new Protestant churches',
 'Protestantism',
 'education of children as Catholics',
 'prohibited emigration',
 'Four thousand',
 '"new converts"',
 'Holland, Prussia, and South Africa',
 'Switzerland and the Netherlands',
 '1555',
 'France Antarctique',
 '1560',
 'the Guanabara Confession